In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
DOMAIN = """
(define (domain item-sorting)
  (:requirements :strips :typing :equality)
  (:types item container)
  (:predicates (on-table ?i - item)
	           (gripper-empty)
	           (gripping ?i - item)
               (in-container ?i - item ?c - container)
 )

  (:action pick-up
	     :parameters (?i - item)
	     :precondition (and (on-table ?i)(gripper-empty))
	     :effect
	     (and (not (on-table ?i))
		   (not (gripper-empty))
		   (gripping ?i)))

  (:action put-down
	     :parameters (?i - item)
	     :precondition (gripping ?i)
	     :effect
	     (and (not (gripping ?i))
		   (gripper-empty)
		   (on-table ?i)))

  (:action put-in-container
	     :parameters (?i - item ?c - container)
	     :precondition (and (gripping ?i))
	     :effect
	     (and (not (gripping ?i))
		   (gripper-empty)
		   (in-container ?i ?c)))

  (:action take-from-container
	     :parameters (?i - item ?c - container)
	     :precondition (and (in-container ?i ?c)(gripper-empty))
	     :effect
	     (and (gripping ?i)
		   (not (gripper-empty))
		   (not (in-container ?i ?c)))))
"""

In [ ]:
PROBLEM = """
(define (problem grocery-bagging)

(:domain item-sorting)
(:objects
milk-carton lemon green-bottle loaf-of-bread red-box-of-cereal red-can-of-soda - item
dark-wood-container light-wood-container - container)

(:init

(gripper-empty)

(on-table milk-carton)
(on-table lemon)
(on-table green-bottle)
(on-table loaf-of-bread)
(on-table red-box-of-cereal)
(on-table red-can-of-soda)

)

(:goal (and
(in-container milk-carton dark-wood-container)
))

)
"""


In [ ]:
from semantic_state_estimator.utils.pddl2nl_query_converter import PDDL2NLQueryConverter

# model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
model_id = "meta-llama/Meta-Llama-3-70B-Instruct"
# model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# model_id = "meta-llama/Meta-Llama-3.1-70B-Instruct"

converter = PDDL2NLQueryConverter.from_uninitialized(model_id, DOMAIN, PROBLEM)


In [ ]:
out = converter.convert_to_nl(converter.all_grounded_predicates)

In [ ]:
out

In [ ]:
import tempfile
import os

import unified_planning as up
from unified_planning.io import PDDLReader

def create_up_problem_from_ppdl_str(domain_str, problem_str):
    reader = PDDLReader()
    
    with tempfile.NamedTemporaryFile(mode='w', delete=False) as f_dom:
        with tempfile.NamedTemporaryFile(mode='w', delete=False) as f_prob:
            f_dom.write(domain_str)
            f_prob.write(problem_str)

            f_dom.close()
            f_prob.close()
        
            up_problem = reader.parse_problem(f_dom.name, f_prob.name)
            
            os.remove(f_dom.name)
            os.remove(f_prob.name)

    return up_problem

problem = create_up_problem_from_ppdl_str(DOMAIN, PROBLEM)

In [ ]:
types = problem.user_types
types

In [ ]:
problem.fluents

In [ ]:
objects = {}
for t in types:
    objects[t.name] = list(problem.objects(t))

objects

In [ ]:
predicates = problem.fluents
predicates[0], predicates[0].arity, predicates[0].signature, predicates[0].signature[0].type

In [ ]:
import transformers
import torch

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# model_id = "meta-llama/Meta-Llama-3.1-70B-Instruct"

pipeline = transformers.pipeline(
  "text-generation",
  model=model_id,
  model_kwargs={"torch_dtype": torch.bfloat16},
  device_map="auto"
)

In [ ]:
dialogs = [
        [{"role": "user", "content": "what is the recipe of mayonnaise?"}],
        [
            {"role": "user", "content": "I am going to Paris, what should I see?"},
            {
                "role": "assistant",
                "content": """\
Paris, the capital of France, is known for its stunning architecture, art museums, historical landmarks, and romantic atmosphere. Here are some of the top attractions to see in Paris:

1. The Eiffel Tower: The iconic Eiffel Tower is one of the most recognizable landmarks in the world and offers breathtaking views of the city.
2. The Louvre Museum: The Louvre is one of the world's largest and most famous museums, housing an impressive collection of art and artifacts, including the Mona Lisa.
3. Notre-Dame Cathedral: This beautiful cathedral is one of the most famous landmarks in Paris and is known for its Gothic architecture and stunning stained glass windows.

These are just a few of the many attractions that Paris has to offer. With so much to see and do, it's no wonder that Paris is one of the most popular tourist destinations in the world.""",
            },
            {"role": "user", "content": "What is so great about #1?"},
        ],
        [
            {"role": "system", "content": "Always answer with Haiku"},
            {"role": "user", "content": "I am going to Paris, what should I see?"},
        ],
        [
            {
                "role": "system",
                "content": "Always answer with emojis",
            },
            {"role": "user", "content": "How to go from Beijing to NY?"},
        ],
    ]

prompts = [pipeline.tokenizer.apply_chat_template(d, tokenize=False) for d in dialogs]

In [ ]:
responses = pipeline(prompts,
                     max_new_tokens=512,
                     eos_token_id=[
                         pipeline.tokenizer.eos_token_id,
                         pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
                     ],
                     pad_token_id = pipeline.tokenizer.eos_token_id,
                     do_sample=True,
                     temperature=0.6,
                     top_p=0.9)

In [ ]:
def print_chat_result(dialog, result, prompt_len):
    for msg in dialog:
        print(f"{msg['role'].capitalize()}: {msg['content']}\n")
    print(
        f"> ASSISTANT: {result[0]['generated_text'][prompt_len:]}"
    )
    print("\n==================================\n")

for i, (d, result) in enumerate(zip(dialogs, responses)):
    print_chat_result(d, result, len(prompts[i]))

In [ ]:
objects_by_type = "\n".join([f'{key} type: {list(map(str, value))}' for key, value in objects.items()])

system = f"""The following is a PDDL domain
{DOMAIN}
Here are the names of all the objects in the current problem, sorted by their type:
{objects_by_type}
Given a grounded predicate with concrete variables, write a natural language yes-no query whose answer determines the truth value of the predicate.
Respond only with this natural language query and nothing else.
"""

# query = "on-table(milk-carton)"
# query = "gripper-empty"
# query = "gripping(loaf-of-bread)"
query = "in-container(red-soda-can, light-wood-container)"

dialog = [{"role": "system", "content": system}, {"role": "user", "content": query}]
prompt = pipeline.tokenizer.apply_chat_template(dialog, tokenize=False)

In [ ]:
response = pipeline(
    prompt,
    max_new_tokens=512,
    eos_token_id=[
        pipeline.tokenizer.eos_token_id,
        pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ],
    pad_token_id=pipeline.tokenizer.eos_token_id,
    do_sample=True,
    temperature=0.6,
    top_p=0.9
)
print_chat_result(dialog, response, len(prompt))